# S2DR4 for Biak burn scars — 1 m chips for SAM

Runs Gamma Earth's S2DR4 on two burning events from the Biak hotspot project and downloads the 1 m GeoTIFFs.

**Colab runs Python 3.13 since 2026-08-19; S2DR4 needs Linux x86-64, Python 3.12 and a CUDA GPU** (gamma.earth/products). So the notebook builds a Python 3.12 virtual environment with `uv` and runs S2DR4 inside it as a subprocess. The notebook's own kernel stays on 3.13 and is used only for plotting and download.

**Date: 2026-08-28.** Three days after the last detection of both events, and the clearest Sentinel-2 acquisition in the post-event window: Cloud Score+ says 100% clear over E0313 and 88% over E0368 (checked in Earth Engine on 2026-09-27).

| event | centre (lon, lat) | detection bbox | dNBR+ area (Task 18) |
|---|---|---|---|
| E0313 | 136.039655, -1.134455 | 1.93 x 2.60 km | 234 ha |
| E0368 | 136.234230, -1.152465 | 1.64 x 0.89 km | 82 ha |

**Credits.** Region C bills 1 credit per 2 km². The tile size `test()` processes around a point is not documented, so **run E0313 first, read the tile size printed in step 5, and only then decide whether to run E0368.**

**Before running:** Runtime → Change runtime type → **GPU**.

The 1 m detail is model output, not observation. It is for delineation and visual interpretation; burned-area figures stay on the native 10/20 m dNBR+ (F11).

## 1. Check the runtime

A GPU is required. If `cuda: False`, switch the runtime type before going on.

In [ ]:
import sys
print('notebook python', sys.version.split()[0])
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print('cuda:', torch.cuda.is_available())

## 2. Build the Python 3.12 environment (safe to re-run)

Colab runs Python 3.13; S2DR4 needs Linux x86-64, **Python 3.12** and a CUDA GPU. This cell builds that environment and installs S2DR4 into it. Later cells call it through `RUN`.

**It checks first and does nothing if the environment already works**, so after a re-upload of the notebook, or a fresh kernel on a live runtime, just run this one cell: it restores `PY`, `OUT` and `RUN` in seconds without reinstalling anything.

It also handles the three things that broke earlier: the GDAL bindings must be built against numpy (else `_gdal_array` is missing), the wheel's metadata omits dependencies such as `tqdm`, and two Colab environment variables have to be overridden.

In [ ]:
import os, re, subprocess

PY = '/content/py312/bin/python'
OUT = '/content/output'
WHEEL = 'https://storage.googleapis.com/0x7ff601307fa5/s2dr4-20260126.1-cp312-cp312-linux_x86_64.whl'
GIRDER = 'https://girder.github.io/large_image_wheels'

# Two Colab quirks, fixed through the environment of every call into the 3.12 env:
#   LD_PRELOAD  - GDAL's compiled module needs libstdc++ loaded first, or the import
#                 dies with "cannot allocate memory in static TLS block".
#   MPLBACKEND  - Colab exports 'module://matplotlib_inline.backend_inline', which this
#                 environment does not have, and matplotlib then refuses to import.
ENV = {**os.environ, 'MPLBACKEND': 'Agg', 'LD_PRELOAD': '/lib/x86_64-linux-gnu/libstdc++.so.6'}
RUN = 'MPLBACKEND=Agg LD_PRELOAD=/lib/x86_64-linux-gnu/libstdc++.so.6 ' + PY  # for ! lines

CHECK = ('import sys, s2dr4, s2dr4.inferutils, torch; '
         'print("py", sys.version.split()[0], "| s2dr4", getattr(s2dr4, "__version__", "unknown"), '
         '"| torch", torch.__version__, "| cuda", torch.cuda.is_available())')
PIP_NAME = {'cv2': 'opencv-python-headless', 'skimage': 'scikit-image', 'PIL': 'pillow',
            'sklearn': 'scikit-learn', 'yaml': 'pyyaml', 'osgeo': 'gdal'}


def sh(cmd):
    return subprocess.run(cmd, shell=True, env=ENV, text=True, capture_output=True)


def imports():
    """(ok, log) for importing s2dr4 in the 3.12 environment.

    The interpreter may not exist yet: a fresh Colab runtime has no /content/py312,
    and calling it then raises FileNotFoundError instead of reporting a failure."""
    if not os.path.exists(PY):
        return False, 'no interpreter at ' + PY
    p = subprocess.run([PY, '-c', CHECK], capture_output=True, text=True, env=ENV)
    return p.returncode == 0, p.stdout + p.stderr


ok, log = imports()
if ok:
    print(log.strip())
    print('environment already built — nothing to install.')
else:
    print('building the Python 3.12 environment (several minutes, a few GB)...')
    sh('apt-get -qq install -y libgdal-dev gdal-bin')
    gdal_version = sh('gdal-config --version').stdout.strip()
    print('system libgdal:', gdal_version)

    sh('pip -q install uv')
    if not os.path.exists(PY):
        print(sh(f'uv venv --python 3.12 /content/py312').stderr.strip())

    # numpy first, then the GDAL bindings built against it: with build isolation the
    # gdal_array extension is silently skipped and arosics later fails on _gdal_array.
    sh(f'uv pip install --python {PY} numpy setuptools wheel')
    sh(f'uv pip install --python {PY} --no-build-isolation "gdal=={gdal_version}"')
    if sh(f'{PY} -c "from osgeo import gdal_array"').returncode != 0:
        print('gdal_array missing — falling back to prebuilt GDAL wheels')
        sh(f'uv pip install --python {PY} --reinstall --find-links {GIRDER} GDAL')

    sh(f'uv pip install --python {PY} {WHEEL} earthengine-api tqdm')

    # the wheel's metadata is incomplete; install whatever the import still asks for
    for attempt in range(12):
        ok, log = imports()
        if ok:
            break
        missing = re.search(r"No module named '([\w.]+)'", log)
        if not missing:
            break
        mod = missing.group(1).split('.')[0]
        name = PIP_NAME.get(mod, mod)
        print('[%d] missing %s -> installing %s' % (attempt + 1, mod, name))
        sh(f'uv pip install --python {PY} {name}')

    print(log.strip())
    print('s2dr4 ready' if ok else 'still failing — paste the log above')


## 3. Earth Engine sign-in

Authenticate once in the notebook kernel. The credentials land in `~/.config/earthengine/credentials`, which the 3.12 environment reads as well. Use the same Google account as the project (`deeplearning-370412`).

If S2DR4 turns out to need no Earth Engine access of its own, this cell costs nothing.

In [ ]:
import ee
PROJECT = 'deeplearning-370412'
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print('notebook kernel ee ok:', ee.Number(1).getInfo() == 1)
!{RUN} -c "import ee; ee.Initialize(project='{PROJECT}'); print('py312 ee ok:', ee.Number(1).getInfo() == 1)" || echo 'py312 could not reach Earth Engine — see the note in step 4'

## 4. Run E0313 (Yendidori, 2026-08-28)

The runner script is written to disk and executed by the 3.12 interpreter. Takes several minutes on a GPU.

If it fails on Earth Engine credentials, run `!{PY} -m ee authenticate --quiet` and repeat.

In [ ]:
%%writefile /content/run_s2dr4.py
import sys, time
import s2dr4.inferutils

lon, lat, date = float(sys.argv[1]), float(sys.argv[2]), sys.argv[3]
t0 = time.time()
s2dr4.inferutils.test((lon, lat), date)
print('elapsed %.1f min' % ((time.time() - t0) / 60))

In [ ]:
!{RUN} /content/run_s2dr4.py 136.039655 -1.134455 2026-08-28
!ls -lhR /content/output 2>/dev/null || ls -lh /content

## 5. What did it actually process, and what did it cost?

Prints the tile extent in km² and the credit cost at Region C (1 credit = 2 km²). **Read this before running the second event.** If the output landed somewhere other than `/content/output`, fix `OUT` below.

In [ ]:
import glob, rasterio
OUT = '/content/output'

paths = sorted(glob.glob(f'{OUT}/**/*.tif*', recursive=True))
print('files:', len(paths))
for path in paths:
    with rasterio.open(path) as r:
        px = abs(r.transform.a)
        km2 = (r.width * px / 1000) * (r.height * px / 1000)
        print(f"{path.split('/')[-1]:45s} {r.width}x{r.height} px  {px:.2f} m  "
              f"{r.count:2d} bands  {r.dtypes[0]:8s} {km2:6.2f} km2  ~{km2/2:.1f} credits (region C)")
        print('   crs', r.crs, ' bounds', [round(b, 5) for b in r.bounds], ' descriptions', r.descriptions)

## 6. Quick look: true colour and SWIR false colour

Burn scars read most clearly in B12/B8A/B4. The band order is printed in step 5 rather than assumed — adjust `idx` if the descriptions disagree.

In [ ]:
import glob, numpy as np, rasterio, matplotlib.pyplot as plt

# The MS file holds 10 bands, not 12: S2DR4 drops B1 and B9/B10, so the order is
# B2 B3 B4 B5 B6 B7 B8 B8A B11 B12. The first plot proved it - reading index 3 as
# B4 produced an infrared-looking image, because index 3 is actually B5.
IDX = {'B2': 0, 'B3': 1, 'B4': 2, 'B5': 3, 'B6': 4, 'B7': 5, 'B8': 6, 'B8A': 7,
       'B11': 8, 'B12': 9}

ms = [p for p in sorted(glob.glob(f'{OUT}/**/*_MS.tif', recursive=True))]
print('multispectral files:', ms)

with rasterio.open(ms[0]) as r:
    a = r.read().astype('float32')
print('bands:', a.shape[0])

def stretch(bands):
    return np.dstack([np.clip((b - np.percentile(b, 2)) /
                              max(np.percentile(b, 98) - np.percentile(b, 2), 1e-6), 0, 1)
                      for b in bands])

fig, ax = plt.subplots(1, 3, figsize=(24, 8))
ax[0].imshow(stretch([a[IDX['B4']], a[IDX['B3']], a[IDX['B2']]]))
ax[0].set_title('true colour 1 m (B4 B3 B2)')
ax[1].imshow(stretch([a[IDX['B12']], a[IDX['B8A']], a[IDX['B4']]]))
ax[1].set_title('SWIR false colour 1 m (B12 B8A B4) - burn scars')
ax[2].imshow(stretch([a[IDX['B8']], a[IDX['B4']], a[IDX['B3']]]))
ax[2].set_title('colour infrared 1 m (B8 B4 B3)')
for x in ax:
    x.axis('off')
plt.tight_layout()
plt.show()


## 7. The other two events

Each run processes a fixed **4100 x 4100 px tile at 1 m = 16.81 km²** around the point, whatever the size of the event. Through the Colab free-scene route this does not draw on the Gamma Earth credit balance; the free tier is counted in scenes (20 at the time of writing).

- **E0368** (Insumarires): the 2026-08-28 scene is 88% clear here, so some cloud may fall in the tile.
- **E0301** (Anjareuw): the largest event in the record, 142 detections and 500 ha of dNBR+ change, and the one worth the most in the comparison. Its detections span 3.4 x 2.9 km, so they sit inside a single tile.

In [ ]:
!{RUN} /content/run_s2dr4.py 136.234230 -1.152465 2026-08-28   # E0368
!ls -lhR /content/output


In [ ]:
!{RUN} /content/run_s2dr4.py 136.142385 -1.177770 2026-08-28   # E0301
!ls -lhR /content/output


## 7b. The same three tiles on 23 August 2026

The 28 August scene is after the burning. The **23 August** scene is in the middle of it: all three events ran to 25 August, and a scar burnt a day or two earlier still reads bright orange there, where by 28 August it has faded to dull brown. The field survey used 23 August as its basemap for the same reason, it being the clearest scene of the episode.

Both dates earn their place in the labelling project: 23 August shows what was burning, 28 August shows what remained. The native 20 m versions of both are already in the QGIS project; these runs add the 1 m detail.

Three more scenes off the free tier. The output lands beside the 28 August files, under its own date, so nothing is overwritten.

In [ ]:
DATE_MID = '2026-08-23'

# Write the runner here too, so this cell does not depend on section 4 having
# been run: a fresh runtime has no /content/run_s2dr4.py.
from pathlib import Path
Path('/content/run_s2dr4.py').write_text('''import sys, time
import s2dr4.inferutils

lon, lat, date = float(sys.argv[1]), float(sys.argv[2]), sys.argv[3]
t0 = time.time()
s2dr4.inferutils.test((lon, lat), date)
print('elapsed %.1f min' % ((time.time() - t0) / 60))
''')

!{RUN} /content/run_s2dr4.py 136.039655 -1.134455 {DATE_MID}   # E0313 Yendidori
!{RUN} /content/run_s2dr4.py 136.234230 -1.152465 {DATE_MID}   # E0368 Insumarires
!{RUN} /content/run_s2dr4.py 136.142385 -1.177770 {DATE_MID}   # E0301 Anjareuw
!ls -lhR /content/output


## 7c. Why did 23 August come back as 25 August?

The date reaches the wheel intact: the log's first line reads `Preprocessing T53MPU-...-20260823`. The scene exists in the catalogue S2DR4 actually reads, which is the Element84 STAC on AWS and not Earth Engine (`fetch_S2L2A_STAC_aoi`, `pystac_client`, `stackstac` are in the compiled module). Yet all three tiles produced 25 August.

Two guesses died already. It is not missing data, and it is not that S2C scenes are ignored outright, because 13 and 3 August are S2C and both appear in the reference stacks.

This cell costs one scene and splits what is left: ask for a date whose own acquisition is S2C, and see which date comes back.

In [ ]:
# One tile, one scene: does S2DR4 ever target a Sentinel-2C acquisition?
#
# 20 August 2026 over tile 53MPU is S2C (cloud 27.7%). Its neighbours are
# 18 August S2B and 23 August S2C. The Element84 STAC catalogue S2DR4 reads
# holds all three, checked on 2026-10-08.
#
# Read the last "Processing T53MPU-..." line of the log:
#   -20260820  the date was honoured, and the 23 August substitution had some
#              other cause
#   -20260818  it fell back to the nearest Sentinel-2B scene, so S2C is never
#              a target even though S2C scenes do appear in the reference
#              stacks (13 and 3 August did)
#   anything else  paste the log, the explanation is neither of those

from pathlib import Path
Path('/content/run_s2dr4.py').write_text('''import sys, time
import s2dr4.inferutils

lon, lat, date = float(sys.argv[1]), float(sys.argv[2]), sys.argv[3]
t0 = time.time()
s2dr4.inferutils.test((lon, lat), date)
print('elapsed %.1f min' % ((time.time() - t0) / 60))
''')

!{RUN} /content/run_s2dr4.py 136.039655 -1.134455 2026-08-20   # E0313 Yendidori
!ls -lh /content/output/ID/T53MPU/*/


## 7d. The other two tiles on 23 August

The probe answered its question and handed over a bonus: asking for 20 August returned **23 August**, the scene originally wanted, for E0313. Both earlier guesses are dead, since 23 August is Sentinel-2C and S2DR4 clearly can target it.

What the package does with a requested date stays undocumented. This cell takes the workaround and finishes the other two tiles.

In [ ]:
# The other two tiles on 23 August, by asking for 20 August.
#
# Asking for 23 August returned 25 August on all three tiles. Asking for
# 20 August returned 23 August, which is the scene wanted: Sentinel-2C,
# 0.4% cloud over the tile, the clearest of the episode and the basemap the
# field survey used. Why the package substitutes a date is not documented and
# not visible in the compiled module, so this works around it rather than
# explaining it.
#
# Check the "Processing T53MPU-..." line of each run: it should end -20260823.

from pathlib import Path
Path('/content/run_s2dr4.py').write_text('''import sys, time
import s2dr4.inferutils

lon, lat, date = float(sys.argv[1]), float(sys.argv[2]), sys.argv[3]
t0 = time.time()
s2dr4.inferutils.test((lon, lat), date)
print('elapsed %.1f min' % ((time.time() - t0) / 60))
''')

!{RUN} /content/run_s2dr4.py 136.234230 -1.152465 2026-08-20   # E0368 Insumarires
!{RUN} /content/run_s2dr4.py 136.142385 -1.177770 2026-08-20   # E0301 Anjareuw
!ls -lh /content/output/ID/T53MPU/*/


## 8. Download

Zips everything and downloads it. Unzip into `C:\\Users\\User\\Documents\\Biak_hotspot\\data\\raw\\s2dr4\\` — that path is gitignored, and the files are model output, so they stay out of the repo.

In [ ]:
from google.colab import files
!cd /content && zip -qr s2dr4_biak_tiles.zip output
!ls -lh /content/s2dr4_biak_tiles.zip
files.download('/content/s2dr4_biak_tiles.zip')